# NB_SETUP_LEGACY_GOLD_POI

Init một lần từ gold cũ `sgr_visitvn_prd_batch` sang map và hai bảng `gld_srv_*` trên `sgr_visitvn_dev`.
Chỉ INSERT. Dòng đã có khóa thì bỏ qua. Mỗi dòng ghi `load_origin = LEGACY_INIT`.
`p_mode = PLAN` không ghi. Không gắn lịch 15 phút.


In [ ]:
p_mode = "PLAN"
p_pin_json = ""


In [ ]:
%run NB_LIB_TRANSFORM_SLV_GLD


In [ ]:
OLD = "sgr_visitvn_prd_batch.lh_vv_gold.dbo"
MAP_FQ = "sgr_visitvn_dev.lh_vv_gold.dbo.gld_poi_legacy_id_map"
INFO_FQ = "sgr_visitvn_dev.lh_vv_gold.dbo.gld_srv_poi_info"
ML_FQ = "sgr_visitvn_dev.lh_vv_gold.dbo.gld_srv_poi_multi_lang"
REV_FQ = "sgr_visitvn_dev.lh_vv_gold.dbo.gld_srv_poi_review"
AMENITY_FQ = "sgr_visitvn_dev.lh_vv_gold.dbo.gld_srv_poi_amenity"
MEDIA_FQ = "sgr_visitvn_dev.lh_vv_gold.dbo.gld_srv_poi_media_gallery"
DICT_FQ = "sgr_visitvn_dev.lh_vv_silver.dbo.ref_dictionary_amenities"
PL = "PL_VV_TRANS_SLV_TO_GLD_30M"


In [ ]:
ORIGIN = "LEGACY_INIT"

def legacy_source_name(c):
    """google -> google ; partner_portal -> partner ; khác -> NULL (không tự đặt)."""
    return F.when(c == F.lit("google"), F.lit("google")).when(c == F.lit("partner_portal"), F.lit("partner"))

def new_poi_id(source_name, source_id):
    """poi_id model mới; không lang; source_id NULL / trống -> NULL (uuid_md5 không tự chặn chuỗi trống)."""
    ok = source_id.isNotNull() & (F.trim(source_id) != F.lit(""))
    return (F.when(ok & (source_name == F.lit("google")), uuid_md5(F.lit("google"), source_id))
             .when(ok & (source_name == F.lit("partner")), uuid_md5(source_id, F.lit("partner"))))

def insert_only_sql(fq: str, view: str, cols: list, pk: tuple) -> str:
    """MERGE chỉ INSERT: khoá đã có (sống hoặc xoá mềm) thì bỏ qua."""
    on = " AND ".join(f"t.`{c}` = s.`{c}`" for c in pk)
    ins = ", ".join(f"`{c}`" for c in cols)
    val = ", ".join(f"s.`{c}`" for c in cols)
    return (f"MERGE INTO {quote_name(fq)} AS t\nUSING {view} AS s\nON {on}\n"
            f"WHEN NOT MATCHED THEN INSERT ({ins})\nVALUES ({val})")

def single_payload(df, keys: list, payload: list, name: str = "_p"):
    """Gom theo khoá đích. (khoá có đúng 1 payload, khoá có > 1 payload). Không chọn winner."""
    g = df.groupBy(*keys).agg(F.collect_set(F.struct(*payload)).alias(name))
    return g.where(F.size(name) == 1), g.where(F.size(name) > 1)

def legacy_upsert_sql(fq: str, view: str, cols: list, pk: tuple, claim_null: bool = False) -> str:
    """INSERT khóa mới. UPDATE chỉ dòng load_origin = LEGACY_INIT. Không sửa dòng pipeline."""
    on = " AND ".join(f"t.`{c}` = s.`{c}`" for c in pk)
    mutable = [c for c in cols if c not in pk and c not in ("created_at", "deleted_at", "load_origin")]
    sets = ", ".join(f"t.`{c}` = s.`{c}`" for c in mutable)
    ins = ", ".join(f"`{c}`" for c in cols)
    val = ", ".join(f"s.`{c}`" for c in cols)
    claim = ""
    if claim_null:
        claimed = (sets + ", " if sets else "") + "t.`load_origin` = 'LEGACY_INIT'"
        claim = f"WHEN MATCHED AND t.`load_origin` IS NULL AND t.`row_hash` = s.`row_hash` THEN UPDATE SET {claimed}\n"
    return (
        f"MERGE INTO {quote_name(fq)} AS t\nUSING {view} AS s\nON {on}\n"
        f"WHEN MATCHED AND t.`load_origin` = 'LEGACY_INIT' THEN UPDATE SET {sets}\n"
        f"{claim}"
        f"WHEN NOT MATCHED THEN INSERT ({ins})\nVALUES ({val})"
    )

def amenity_enum_values(raw):
    """Mỗi enum thành 'giá_trị\\x1fpath'. Cờ boolean true dùng path làm khóa từ điển."""
    import json as _json
    if raw is None:
        return []
    text = str(raw).strip()
    if not text:
        return []
    try:
        node = _json.loads(text)
    except (TypeError, ValueError):
        return []
    found = []
    meta = {"enum", "item_type", "required", "type", "multiple_lang"}

    def add(token, path):
        if isinstance(token, bool):
            token = "true" if token else ""
        if not isinstance(token, str) or not token.strip():
            return
        found.append(token.strip() + "\x1f" + path)

    def walk(value, path):
        if isinstance(value, dict):
            enum = value.get("enum")
            if isinstance(enum, list):
                for item in enum:
                    add(item, path)
            for key, child in value.items():
                if key in meta:
                    continue
                child_path = f"{path}.{key}" if path else key
                if child is True:
                    add("true", child_path)
                else:
                    walk(child, child_path)
        elif isinstance(value, list):
            for child in value:
                walk(child, path)

    walk(node, "")
    return list(dict.fromkeys(found))


In [ ]:
import inspect
import json
import uuid

def _workspace_name():
    ctx = notebookutils.runtime.context
    return ctx.get("currentWorkspaceName") or ctx.get("workspaceName")

def _history_version(fq):
    row = spark.sql(f"DESCRIBE HISTORY {quote_name(fq)} LIMIT 1").first()
    return int(row["version"])

def old(table):
    return spark.sql(
        f"SELECT * FROM {quote_name(OLD + '.' + table)} VERSION AS OF {int(PINS[table])}"
    )

def present_counts(src, fq, pk):
    live = spark.table(fq).select(*[F.col(c) for c in pk], F.col("deleted_at")).withColumn("_hit", F.lit(1))
    joined = src.select(*pk).dropDuplicates(list(pk)).join(live, list(pk), "left")
    row = joined.agg(
        F.sum(F.when(F.col("_hit").isNotNull() & F.col("deleted_at").isNull(), 1).otherwise(0)).alias("live"),
        F.sum(F.when(F.col("_hit").isNotNull() & F.col("deleted_at").isNotNull(), 1).otherwise(0)).alias("deleted"),
    ).first()
    return int(row["live"] or 0), int(row["deleted"] or 0)

def to_target(src, fq, batch_ts):
    """Khung đúng cột bảng đích: cột không nạp = NULL đúng kiểu; row_hash lib; LEGACY_INIT."""
    cols = table_columns(fq)
    biz = [c for c in cols if c not in TECH_NAMES]
    df = src.select(*[(F.col(c) if c in src.columns else F.lit(None)).cast(cols[c]).alias(c) for c in biz])
    df = df.withColumn("row_hash", _hash_expr(df, biz))
    ts = F.lit(batch_ts).cast("timestamp")
    return (df.withColumn("created_at", ts).withColumn("updated_at", ts)
              .withColumn("deleted_at", F.lit(None).cast("timestamp")).withColumn(OWNER_COL, F.lit(ORIGIN)))

def _metrics(raw):
    """operationMetrics có thể là dict hoặc chuỗi JSON. Không gọi json.loads trên dict."""
    if raw is None or raw == "":
        return {}
    if isinstance(raw, dict):
        return raw
    if isinstance(raw, str):
        parsed = json.loads(raw)
        return parsed if isinstance(parsed, dict) else {}
    return {}


def _commit_lookup(fq, v_pre):
    """Đối soát commit theo tag LEGACY_INIT và version trước ghi. Mọi lỗi history -> UNKNOWN."""
    result = {"v_pre": v_pre, "v_post": None, "inserted": None, "commit": "UNKNOWN"}
    try:
        hist = spark.sql(f"DESCRIBE HISTORY {quote_name(fq)} LIMIT 50").collect()
        for row in hist:
            meta = row["userMetadata"] or ""
            if meta == f"LEGACY_INIT:{EXEC}" and int(row["version"]) > int(v_pre):
                metrics = _metrics(row["operationMetrics"])
                result.update(v_post=int(row["version"]),
                              inserted=int(metrics.get("numTargetRowsInserted") or 0),
                              commit="COMMITTED")
                return result
        if hist and int(hist[0]["version"]) == int(v_pre):
            result.update(v_post=int(v_pre), inserted=0, commit="NOT_COMMITTED")
    except Exception as exc:
        result["history_error"] = str(exc)[:200]
    return result


def write_insert_only(fq, df, pk, tag, slot, upsert=False, claim_null=False):
    if not fq.startswith("sgr_visitvn_dev."):
        raise RuntimeError(f"Đích không thuộc sgr_visitvn_dev: {fq}")
    dup = df.groupBy(*pk).count().where("count > 1").count()
    if dup:
        raise RuntimeError(f"{fq} có {dup} khoá trùng payload trước MERGE")
    refresh_flow_lock(PL, EXEC)
    v_pre = _history_version(fq)
    view = f"_v_lgp_{tag}"
    df.createOrReplaceTempView(view)
    spark.conf.set("spark.databricks.delta.commitInfo.userMetadata", f"LEGACY_INIT:{EXEC}")
    write_error = None
    try:
        sql = legacy_upsert_sql(fq, view, df.columns, pk, claim_null) if upsert else insert_only_sql(fq, view, df.columns, pk)
        spark.sql(sql)
    except Exception as exc:
        write_error = exc
    result = _commit_lookup(fq, v_pre)
    for cleanup in (
        lambda: spark.conf.unset("spark.databricks.delta.commitInfo.userMetadata"),
        lambda: spark.catalog.dropTempView(view),
    ):
        try:
            cleanup()
        except Exception as cleanup_exc:
            result["cleanup_error"] = str(cleanup_exc)[:200]
    if slot in summary:
        summary[slot].update(result)
    if write_error is not None:
        raise write_error
    return result

if _workspace_name() != "sgr_visitvn_dev":
    raise RuntimeError("Notebook phải mở ở workspace sgr_visitvn_dev")
if OWNER_COL != "load_origin" or OWNER_COL not in TECH_NAMES or "owner_col" not in inspect.signature(sql_merge_target).parameters:
    raise RuntimeError("Lib chưa có owner_col / OWNER_COL")
if p_mode not in ("PLAN", "RUN"):
    raise RuntimeError("p_mode phải là PLAN hoặc RUN")
for fq in (INFO_FQ, ML_FQ, REV_FQ, AMENITY_FQ, MEDIA_FQ):
    kind = table_columns(fq).get("load_origin")
    if kind != "string":
        raise RuntimeError(f"{fq} chưa có load_origin string: chạy legacyidpy trước")
map_cols = table_columns(MAP_FQ)
for name in ("legacy_poi_id", "lang", "legacy_ingestion_source", "source_id", "source_name", "poi_id", "poi_id_status"):
    if name not in map_cols:
        raise RuntimeError(f"{MAP_FQ} thiếu {name}")

PINS = json.loads(p_pin_json) if p_pin_json else {}
for table in ("poi_master", "poi_enrichment_display", "poi_detail", "poi_review", "poi_rating"):
    PINS[table] = int(PINS[table]) if table in PINS and PINS[table] != "" else _history_version(f"{OLD}.{table}")
pm_pin = old("poi_master")
pin_n = pm_pin.agg(F.count("*").alias("n"), F.countDistinct("poi_id").alias("d")).first()
if int(pin_n["n"]) != int(pin_n["d"]):
    raise RuntimeError("poi_master tại pin có poi_id trùng")

EXEC = "lgp-" + uuid.uuid4().hex
summary = {"exec_id": EXEC, "mode": p_mode, "pins": PINS, "review": {"load_origin_present": table_columns(REV_FQ).get("load_origin") == "string", "inserted": 0}}
status = "SUCCESS"
stage = "preflight"
locked = False
try:
    if p_mode == "RUN":
        ok, who = acquire_flow_lock(PL, EXEC, EXEC, LOCK_TIMEOUT_DEFAULT_MIN)
        if not ok:
            raise RuntimeError(f"Không nhận khoá {PL}: {who}")
        locked = True
    stage = "map"
    pm = old("poi_master")
    map_src = (pm.select(F.col("poi_id").alias("legacy_poi_id"), "lang",
                         F.col("ingestion_source").alias("legacy_ingestion_source"), "source_id")
                 .withColumn("source_name", legacy_source_name(F.col("legacy_ingestion_source")))
                 .withColumn("poi_id", new_poi_id(F.col("source_name"), F.col("source_id")))
                 .withColumn("poi_id_status", F.when(F.col("poi_id").isNotNull(), F.lit("OK"))
                                               .otherwise(F.lit("MISSING_EVIDENCE"))))
    id_cols = ["legacy_poi_id", "lang", "legacy_ingestion_source", "source_id", "source_name", "poi_id", "poi_id_status"]
    bad_key_n = map_src.groupBy("legacy_poi_id").count().where("count > 1").count()
    if bad_key_n:
        raise RuntimeError(f"map nguồn có {bad_key_n} legacy_poi_id trùng")
    stored = spark.table(MAP_FQ).select(*id_cols)
    stored_bad = stored.groupBy("legacy_poi_id").count().where("count > 1").count()
    if stored_bad:
        raise RuntimeError(f"map đã ghi có {stored_bad} legacy_poi_id trùng")
    left = stored.select(*[F.col(c).alias(f"_s_{c}") for c in id_cols])
    right = map_src.select(*[F.col(c).alias(f"_f_{c}") for c in id_cols])
    compared = left.join(right, left["_s_legacy_poi_id"].eqNullSafe(right["_f_legacy_poi_id"]), "full_outer")
    differ = None
    for col_name in id_cols:
        term = ~F.col(f"_s_{col_name}").eqNullSafe(F.col(f"_f_{col_name}"))
        differ = term if differ is None else differ | term
    # Dòng chỉ có ở nguồn là insert mới, không phải drift. Drift chỉ khi dòng đã ghi mất nguồn hoặc lệch field.
    persisted = F.col("_s_legacy_poi_id").isNotNull()
    drift_n = compared.where(persisted & differ).count()
    summary["map"] = {
        "missing_evidence_n": map_src.where("poi_id_status = 'MISSING_EVIDENCE'").count(),
        "candidates": map_src.count(),
        "drift_n": drift_n,
        "bad_key_n": stored_bad,
    }
    if p_mode == "RUN" and drift_n:
        raise RuntimeError(f"drift_n={drift_n}: pin không khớp map đã ghi, dừng trước khi ghi")
    fresh_only = map_src.join(stored.select("legacy_poi_id"), "legacy_poi_id", "left_anti")
    map_rows = stored.unionByName(fresh_only.select(*id_cols))
    if p_mode == "RUN":
        written = write_insert_only(MAP_FQ, map_src, ("legacy_poi_id",), "map", "map")
        summary["map"].update(written)
    else:
        summary["map"].update({"inserted": 0, "v_pre": None, "v_post": None})
    stage = "info"
    mp = map_rows.where("poi_id_status = 'OK'")
    CMP = ["business_sector", "business_category", "subcategory_tags", "operating_status",
           "is_verified", "is_bookable", "rating_overall", "rating_count"]
    g = mp["source_name"] == F.lit("google")
    rows = pm.join(mp, pm["poi_id"] == mp["legacy_poi_id"]).select(
        mp["poi_id"], mp["source_name"], pm["lang"], pm["poi_name"], pm["district"],
        pm["is_active"].alias("_old_active"),
        *[pm[c] for c in CMP[:6]],
        pm["rating_overall"].alias("_raw_rating"), pm["rating_count"].alias("_raw_count"),
        F.when(g, pm["rating_overall"].cast("decimal(4,2)")).alias("rating_overall"),
        F.when(g, pm["rating_count"].cast("int")).alias("rating_count"))
    cast_fail_n = rows.where(
        (F.col("source_name") == F.lit("google")) & (
            (F.col("_raw_rating").isNotNull() & F.col("rating_overall").isNull())
            | (F.col("_raw_count").isNotNull() & F.col("rating_count").isNull())
        )
    ).count()
    if cast_fail_n > 0:
        raise RuntimeError(f"cast_fail_n={cast_fail_n}: dừng trước khi ghi")
    agg = rows.groupBy("poi_id").agg(
        F.collect_set(F.struct("source_name", *CMP)).alias("_cmp"),
        F.collect_set(F.when(F.col("lang") == "vi", F.struct("poi_name", "district"))).alias("_vi"),
        F.max(F.when(F.col("_old_active") == F.lit(False), 1).otherwise(0)).alias("_inactive"))
    keep = (F.size("_cmp") == 1) & (F.size("_vi") <= 1)
    one_vi = F.size("_vi") == 1
    info_src = agg.where(keep).select(
        "poi_id", *[F.col("_cmp").getItem(0)[c].alias(c) for c in ["source_name", *CMP]],
        F.when(one_vi, F.col("_vi").getItem(0)["poi_name"]).alias("poi_name"),
        F.when(one_vi, F.col("_vi").getItem(0)["district"]).alias("district"),
        F.lit(True).alias("is_active"), "_inactive")
    info_live, info_deleted = present_counts(info_src, INFO_FQ, ("poi_id",))
    info_existing = spark.table(INFO_FQ).select("poi_id").dropDuplicates(["poi_id"])
    info_new = info_src.join(info_existing, "poi_id", "left_anti")
    summary["info"] = {
        "groups": agg.count(), "cmp_conflict_n": agg.where(F.size("_cmp") > 1).count(),
        "vi_conflict_n": agg.where(F.size("_vi") > 1).count(),
        "no_vi_n": agg.where(F.size("_vi") == 0).count(),
        "cast_fail_n": cast_fail_n,
        "old_inactive_n": info_src.where("_inactive = 1").count(),
        "candidates": info_src.count(), "present_live": info_live, "present_deleted": info_deleted,
        "to_insert": info_new.count(),
    }
    batch_ts = spark.sql("SELECT current_timestamp() AS ts").first()["ts"]
    parents = (spark.table(INFO_FQ).where("deleted_at IS NULL").select("poi_id")
               .union(info_new.select("poi_id")).dropDuplicates(["poi_id"]))
    detail = old("poi_detail").select(
        "poi_id", "poi_address_json", "poi_contact_json", "poi_content_json",
        "poi_opening_hours_json", "poi_media_json")
    vi_key = (mp.where(F.col("lang") == "vi")
              .select(F.col("legacy_poi_id").alias("poi_id"), F.col("poi_id").alias("_new"), "source_name").dropDuplicates(["_new"]))
    det = detail.join(vi_key, "poi_id", "inner")
    rate = (old("poi_rating").groupBy("poi_id")
            .agg(F.max(F.col("total_reviews").cast("long")).cast("int").alias("review_count")))
    det = det.join(rate, "poi_id", "left")
    def _j(col, path):
        return F.get_json_object(F.col(col), path)
    det_cols = [
        "full_address", "street", "ward", "city", "country_code", "postal_code", "lat", "lng",
        "phone", "email", "website_url", "operating_hours", "holiday_exceptions",
        "cover_id", "cover_blob_url", "review_count"]
    det_rows = det.select(
        F.col("_new").alias("poi_id"),
        _j("poi_address_json", "$.full_address").alias("full_address"),
        _j("poi_address_json", "$.address_line1").alias("street"),
        _j("poi_address_json", "$.ward").alias("ward"),
        _j("poi_address_json", "$.city").alias("city"),
        _j("poi_address_json", "$.country_code").alias("country_code"),
        _j("poi_address_json", "$.postal_code").alias("postal_code"),
        _j("poi_address_json", "$.lat").cast("double").alias("lat"),
        _j("poi_address_json", "$.lng").cast("double").alias("lng"),
        _j("poi_contact_json", "$.phone").alias("phone"),
        _j("poi_contact_json", "$.email").alias("email"),
        _j("poi_contact_json", "$.website_url").alias("website_url"),
        _j("poi_content_json", "$.hours_weekday_text").alias("operating_hours"),
        _j("poi_opening_hours_json", "$.holiday_exceptions").alias("holiday_exceptions"),
        _j("poi_media_json", "$.cover.media_id").alias("cover_id"),
        _j("poi_media_json", "$.cover.blob_url").alias("cover_blob_url"),
        F.col("review_count"),
    )
    det_ok, det_bad = single_payload(det_rows, ["poi_id"], det_cols)
    det_one = det_ok.select("poi_id", *[F.col("_p").getItem(0)[c].alias(c) for c in det_cols])
    summary["info"]["detail_conflict_n"] = det_bad.count()
    info_src = info_src.join(det_one, "poi_id", "left")
    master_cols = ["is_recommended", "destination_id", "destination_name"]
    master_rows = (pm.where(F.col("lang") == "vi")
                   .join(vi_key, pm["poi_id"] == vi_key["poi_id"])
                   .select(
                       vi_key["_new"].alias("poi_id"),
                       pm["is_recommended"],
                       F.get_json_object(pm["destination_ids"], "$[0]").alias("destination_id"),
                       pm["destination_names"].alias("destination_name")))
    master_ok, master_bad = single_payload(master_rows, ["poi_id"], master_cols)
    master_one = master_ok.select(
        "poi_id", *[F.col("_p").getItem(0)[c].alias(c) for c in master_cols])
    summary["info"]["master_conflict_n"] = master_bad.count()
    info_src = info_src.join(master_one, "poi_id", "left")
    if p_mode == "RUN":
        written = write_insert_only(INFO_FQ, to_target(info_src.drop("_inactive"), INFO_FQ, batch_ts), ("poi_id",), "info", "info", True)
        summary["info"].update(written)
    else:
        summary["info"].update({"inserted": 0, "v_pre": None, "v_post": None})
    stage = "multi_lang"
    ML_PAY = ["description", "time_to_spend", "best_time"] + (["cost"] if "cost" in table_columns(ML_FQ) else [])
    display = old("poi_enrichment_display")
    joined = display.join(mp, display["poi_id"] == mp["legacy_poi_id"], "left")
    summary_ml_base = {
        "display_rows": display.count(),
        "orphan_n": joined.where(mp["legacy_poi_id"].isNull()).count(),
        "null_lang_n": display.where("lang IS NULL").count(),
        "lang_mismatch_n": joined.where(mp["legacy_poi_id"].isNotNull() & (display["lang"] != mp["lang"])).count(),
        "cost_loaded": "cost" in ML_PAY,
    }
    pay = ["source_name", *ML_PAY, "experience"]
    picked = joined.where(mp["legacy_poi_id"].isNotNull() & display["lang"].isNotNull()).select(
        mp["poi_id"], mp["lang"], mp["source_name"], *[display[c] for c in ML_PAY],
        display["experiences"].alias("experience"))
    ml_ok, ml_conflict = single_payload(picked, ["poi_id", "lang"], pay)
    ml_src = ml_ok.select("poi_id", "lang", *[F.col("_p").getItem(0)[c].alias(c) for c in pay])
    before_parent = ml_src.count()
    ml_src = ml_src.join(parents, "poi_id", "left_semi")
    ml_live, ml_deleted = present_counts(ml_src, ML_FQ, ("poi_id", "lang"))
    ml_existing = spark.table(ML_FQ).select("poi_id", "lang").dropDuplicates(["poi_id", "lang"])
    ml_new = ml_src.join(ml_existing, ["poi_id", "lang"], "left_anti")
    summary["multi_lang"] = {
        **summary_ml_base, "conflict_n": ml_conflict.count(), "no_parent_n": before_parent - ml_src.count(),
        "candidates": ml_src.count(), "present_live": ml_live, "present_deleted": ml_deleted,
        "to_insert": ml_new.count(),
    }
    if p_mode == "RUN":
        written = write_insert_only(ML_FQ, to_target(ml_src, ML_FQ, batch_ts), ("poi_id", "lang"), "ml", "multi_lang", True)
        summary["multi_lang"].update(written)
    else:
        summary["multi_lang"].update({"inserted": 0, "v_pre": None, "v_post": None})

    stage = "review"
    rating = old("poi_rating").select(
        "poi_id", F.col("overall").cast("decimal(4,2)").alias("_overall"),
        F.col("total_reviews").cast("bigint").alias("_total"))
    reviews = (old("poi_review").alias("rv")
               .join(mp.alias("m"), F.col("rv.poi_id") == F.col("m.legacy_poi_id"))
               .join(rating.alias("rt"), F.col("rv.poi_id") == F.col("rt.poi_id"), "left"))
    rev = reviews.select(
        F.col("rv.review_id").alias("review_id"),
        F.col("m.poi_id").alias("poi_id"),
        F.col("rt._overall").alias("rating_overall"),
        F.col("rt._total").alias("rating_count"),
        F.col("rv.title").alias("title"),
        F.col("rv.review_text").alias("review_text"),
        F.col("rv.rating").cast("decimal(4,2)").alias("rating"),
        F.col("rv.review_date").cast("string").alias("review_date"),
        F.col("rv.reviewer_name").alias("author_name"),
        F.coalesce(F.col("rv.source"), F.col("m.source_name")).alias("source_name"),
    )
    rev_ok, rev_bad = single_payload(
        rev, ["review_id"],
        ["poi_id", "rating_overall", "rating_count", "title", "review_text", "rating", "review_date", "author_name", "source_name"])
    rev_src = rev_ok.select(
        "review_id",
        *[F.col("_p").getItem(0)[c].alias(c) for c in
          ["poi_id", "rating_overall", "rating_count", "title", "review_text", "rating", "review_date", "author_name", "source_name"]])
    summary["review"]["candidates"] = rev_src.count()
    summary["review"]["conflict_n"] = rev_bad.count()
    if p_mode == "RUN":
        written = write_insert_only(REV_FQ, to_target(rev_src, REV_FQ, batch_ts), ("review_id",), "review", "review", True)
        summary["review"].update(written)
    else:
        summary["review"].update({"inserted": 0, "v_pre": None, "v_post": None})

    stage = "media"
    media_schema = "STRUCT<cover:STRUCT<media_id:STRING,blob_url:STRING>,gallery:ARRAY<STRUCT<media_id:STRING,blob_url:STRING>>>"
    parsed = det.select(
        F.col("_new").alias("poi_id"), "source_name",
        F.from_json("poi_media_json", media_schema).alias("parsed"))
    cover = parsed.select(
        "poi_id", "source_name",
        F.col("parsed.cover.media_id").alias("media_id"),
        F.col("parsed.cover.blob_url").alias("blob_url"),
        F.lit(True).alias("is_cover")).where("media_id IS NOT NULL")
    gallery = parsed.select("poi_id", "source_name", F.explode_outer("parsed.gallery").alias("g")).select(
        "poi_id", "source_name",
        F.col("g.media_id").alias("media_id"),
        F.col("g.blob_url").alias("blob_url"),
        F.lit(False).alias("is_cover")).where("media_id IS NOT NULL")
    media_src = (cover.unionByName(gallery)
                 .groupBy("poi_id", "media_id")
                 .agg(F.max(F.col("is_cover").cast("int")).cast("boolean").alias("is_cover"),
                      F.max("blob_url").alias("blob_url"),
                      F.max("source_name").alias("source_name"),
                      F.lit(True).alias("is_active")))
    media_df = to_target(media_src, MEDIA_FQ, batch_ts)
    summary["media"] = {"candidates": media_src.count()}
    if p_mode == "RUN":
        written = write_insert_only(MEDIA_FQ, media_df, ("media_id", "poi_id"), "media", "media", True, True)
        summary["media"].update(written)
    else:
        summary["media"].update({"inserted": 0, "v_pre": None, "v_post": None})

    stage = "amenity"
    from pyspark.sql.types import ArrayType, StringType
    enum_udf = F.udf(amenity_enum_values, ArrayType(StringType()))
    sector = (pm.where(F.col("lang") == "vi")
              .select(F.col("poi_id").alias("legacy_poi_id"), "business_sector"))
    amenity_base = (old("poi_detail")
                    .select(F.col("poi_id").alias("legacy_poi_id"), "amenities_json")
                    .join(vi_key.select(F.col("poi_id").alias("legacy_poi_id"), "_new", "source_name"),
                          "legacy_poi_id", "inner")
                    .join(sector, "legacy_poi_id", "left"))
    enums = (amenity_base.select(
                 F.col("_new").alias("poi_id"), "source_name", "business_sector",
                 F.explode(enum_udf("amenities_json")).alias("_pair"))
             .where(F.col("source_name").isNotNull()
                    & F.col("business_sector").isNotNull()
                    & (F.trim("business_sector") != F.lit("")))
             .select(
                 "poi_id", "source_name", "business_sector",
                 F.lower(F.trim(F.split(F.col("_pair"), "\u001f").getItem(0))).alias("_token"),
                 F.lower(F.trim(F.split(F.col("_pair"), "\u001f").getItem(1))).alias("_path")))
    dictionary = (spark.table(DICT_FQ)
                  .where(F.col("is_active").cast("string").isin("true", "1"))
                  .select(
                      "source_name", "business_sector", "amenity_group", "amenity_raw",
                      "amenity", "amenity_en", "amenity_vi", "service_flags",
                      F.lower(F.trim("business_sector")).alias("_sector"),
                      F.lower(F.trim("amenity_raw")).alias("_raw"),
                      F.lower(F.trim("amenity_en")).alias("_en"),
                      F.lower(F.trim("amenity_vi")).alias("_vi")))
    label = ((F.col("e._token") != F.lit("true"))
             & ((F.col("e._token") == F.col("d._raw"))
                | (F.col("e._token") == F.col("d._en"))
                | (F.col("e._token") == F.col("d._vi"))))
    flag = (F.col("e._token") == F.lit("true")) & (F.col("e._path") == F.col("d._raw"))
    hit = (enums.alias("e").join(
               dictionary.alias("d"),
               (F.col("e.source_name") == F.col("d.source_name"))
               & (F.lower(F.trim(F.col("e.business_sector"))) == F.col("d._sector"))
               & (label | flag),
               "inner")
           .select(
               F.col("e.poi_id").alias("poi_id"),
               F.col("d.business_sector").alias("business_sector"),
               F.col("d.amenity_group").alias("amenity_group"),
               F.col("d.amenity_raw").alias("amenity_raw"),
               F.col("d.amenity").alias("amenity"),
               F.col("d.amenity_en").alias("amenity_en"),
               F.col("d.amenity_vi").alias("amenity_vi"),
               F.col("d.service_flags").alias("service_flags")))
    pay = ["amenity_group", "amenity", "amenity_en", "amenity_vi", "service_flags"]
    am_ok, am_bad = single_payload(hit, ["poi_id", "business_sector", "amenity_raw"], pay)
    am_src = am_ok.select(
        "poi_id", "business_sector", "amenity_raw",
        *[F.col("_p").getItem(0)[c].alias(c) for c in pay])
    if "load_origin" not in table_columns(AMENITY_FQ):
        raise RuntimeError("gld_srv_poi_amenity thiếu load_origin: chạy lại cell legacyidpy")
    summary["amenity"] = {
        "enum_n": enums.count(),
        "candidates": am_src.count(),
        "conflict_n": am_bad.count(),
    }
    if p_mode == "RUN":
        written = write_insert_only(
            AMENITY_FQ, to_target(am_src, AMENITY_FQ, batch_ts),
            ("poi_id", "business_sector", "amenity_group", "amenity_raw"), "amenity", "amenity", True)
        summary["amenity"].update(written)
    else:
        summary["amenity"].update({"inserted": 0, "v_pre": None, "v_post": None})

except Exception:
    status = "FAILED"
    summary["failed_stage"] = stage
    print(json.dumps(summary, default=str))
    raise
finally:
    if locked:
        try:
            release_flow_lock(PL, EXEC, status)
        except Exception as unlock_exc:
            summary["unlock_error"] = str(unlock_exc)[:200]
            print(json.dumps(summary, default=str))
            if status == "SUCCESS":
                raise
print(json.dumps(summary, default=str))
notebookutils.notebook.exit(json.dumps(summary, default=str))
